<a href="https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Ready.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

This feature vector supports a "which page should an editor review first" decision, using the label from w02: a page is declining when trend_direction == "down". Features are built from signals that exist independent of that outcome — traffic, engagement, content metadata, and time-since-update — never from the trend itself.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Label (same definition as w02 — keep it consistent across the project)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Numeric features — fill missing with 0 (documented in Section 2 below)
numeric_features = [
    "content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "engagement_rate", "scroll_rate",
    "word_count", "search_volume", "cpc", "ai_traffic_pct"
]
for col in numeric_features:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

# Categorical features — one-hot encode
categorical_features = ["content_type", "main_intent", "competition_level"]
df_encoded = pd.get_dummies(df, columns=categorical_features, prefix=categorical_features)

feature_cols = numeric_features + [
    c for c in df_encoded.columns
    if any(c.startswith(p + "_") for p in categorical_features)
]

X = df_encoded[feature_cols]
y = df_encoded["is_declining_label"]

print(f"Feature vector: {X.shape[0]} rows, {X.shape[1]} columns")
print(f"Declining rate: {y.mean():.3f}")


Feature vector: 30000 rows, 21 columns
Declining rate: 0.542


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| content_age_days | Days since page first published | Filled 0 | Yes |
| days_since_last_update | Days since last content edit | Filled 0 | Yes |
| impressions_90d | Search impressions, trailing 90 days | Filled 0 | Yes |
| avg_position | Average SERP position | Filled 0 | Yes |
| ctr | Click-through rate | Filled 0 | Yes |
| engagement_rate | On-page engagement signal | Filled 0 | Yes |
| scroll_rate | Scroll depth signal | Filled 0 | Yes |
| word_count | Content length | Filled 0 | Yes |
| search_volume | Keyword search volume | Filled 0 | Yes |
| cpc | Keyword cost-per-click | Filled 0 | Yes |
| ai_traffic_pct | Share of traffic from AI referrals | Filled 0 | Yes |
| content_type, main_intent, competition_level | Categorical metadata | One-hot, missing → no dummy fires | Yes |

All features describe the page's state independent of the trend outcome — none are computed from trend_direction or trend_pct.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Total feature columns: {X.shape[1]}")
print(X.dtypes.value_counts())
print("\nMissing values remaining after fill:")
print(X.isna().sum().sum(), "(should be 0)")

Total feature columns: 21
bool       10
float64     8
int64       3
Name: count, dtype: int64

Missing values remaining after fill:
0 (should be 0)


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

trend_pct and trend_direction are excluded on principle — they define the label directly. To confirm the remaining features aren't secretly encoding the same thing, I checked each feature's correlation with trend_pct and re-ran the depth-2 tree test from w02 with clean features only, watching for a suspicious jump in Precision@50.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Check correlation of every numeric feature with the raw trend_pct (the label source)
corrs = df[numeric_features].corrwith(df["trend_pct"]).sort_values(key=abs, ascending=False)
print("Correlation with trend_pct (label source):")
print(corrs.round(3))

# Clean leakage test: does a normal tree already look "too good"?
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X, y)
score = tree.predict_proba(X)[:, 1]
print(f"\nClean feature Precision@50: {precision_at_k(score, y, 50):.3f}")
print("(w02's leaky version hit 1.000 — a clean, honest score should sit well below that.)")


Correlation with trend_pct (label source):
avg_position              0.047
impressions_90d           0.024
days_since_last_update   -0.014
word_count               -0.012
engagement_rate           0.008
ctr                       0.008
cpc                       0.005
ai_traffic_pct           -0.004
search_volume             0.002
scroll_rate               0.002
content_age_days          0.001
dtype: float64

Clean feature Precision@50: 0.600
(w02's leaky version hit 1.000 — a clean, honest score should sit well below that.)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- trend_pct — the exact value the label bucket is computed from; pure leakage.
- trend_direction — this IS the label; using it as a feature is circular.
- content_id, client_id — identifiers, not signal; risk of memorizing specific pages/clients instead of learning general patterns.
- impression_tier, position_tier — pre-bucketed versions of impressions_90d/avg_position; redundant with features already included.
- Any internal health-score or "needs fix" flags, if present — these would only exist because someone already made the decision this project is trying to support.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_features = ["trend_pct", "trend_direction", "content_id", "client_id",
                      "impression_tier", "position_tier"]
print("Excluded columns:", excluded_features)
print("\nConfirming none leaked into X:")
leaked = [c for c in excluded_features if c in X.columns]
print("Leaked columns found:", leaked if leaked else "None — clean.")

Excluded columns: ['trend_pct', 'trend_direction', 'content_id', 'client_id', 'impression_tier', 'position_tier']

Confirming none leaked into X:
Leaked columns found: None — clean.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.